In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from langchain_core.output_parsers import StrOutputParser

In [3]:
llm = ChatOpenAI(model="gpt-5-nano")

In [4]:
# create a state

class LLMQA(TypedDict):

    topic : str
    joke : str
    explanation : str

In [7]:
# define function

def llm_joke(state : LLMQA) -> LLMQA:

    que = state['topic']
    prompt = f"give a one liner joke on this topic : {que}"

    ans = llm.invoke(prompt)

    parser = StrOutputParser()
    clean_string = parser.invoke(ans)
    state["joke"] = clean_string

    return state


def llm_explain(state: LLMQA) -> LLMQA:

    joke = state['joke']
    prompt = f"explain me this joke in just 1-2 line : {joke}"
    
    ex = llm.invoke(prompt)

    parser = StrOutputParser()
    clean_string = parser.invoke(ex)
    state["explanation"] = clean_string
    
    return state


In [8]:
# define your graph
graph = StateGraph(LLMQA)

# add nodes to your graph
graph.add_node("llm_joke",llm_joke)
graph.add_node("llm_explain",llm_explain)

# add edges to your graph
graph.add_edge(START,"llm_joke")
graph.add_edge("llm_joke","llm_explain")
graph.add_edge("llm_explain",END)

# compile the graph
workflow = graph.compile()


In [9]:
# execute the graph
query = input("Enter the topic :: ")
initial_state = {"topic" : query}

final_state = workflow.invoke(initial_state)

print(final_state)

{'topic': 'ai', 'joke': 'I asked AI for a joke—got a 404: humor not found.', 'explanation': 'It\'s a tech pun: 404 Not Found is used for missing web pages. The joke says humor is "not found"—the AI couldn\'t load a joke.'}
